In [1]:
'''
    Calculated climos from history output
'''

'\n    Calculated climos from history output\n'

In [2]:
import xarray as xr
import numpy as np
#import xcdat as xcd

import glob as glob
import os as os
import re as re

In [3]:
from distributed import Client
from ncar_jobqueue import NCARCluster

cluster = NCARCluster(account='P93300642',interface='ext', job_extra_directives=[],walltime ='12:00:00')
cluster
cluster.scale(jobs=32)
client = Client(cluster)
client

/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/dask_jobqueue/core.py:285: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.74:34933,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [5]:
''' Case Details '''

#run_name = 'f.cam6_3_161.FLTHIST_ne30.ke.001'
run_name = 'f.cam6_3_161.FLTHIST_ne30.ke.004'
dir0 = '/glade/derecho/scratch/rneale/archive/'

hist_pref = 'h0a'
years = [1995,2004]


dir_in = dir0+run_name+'/atm/hist/'
dir_out = dir0+run_name+'/climo/'

# Sort files 
files_unsorted = os.path.join(dir_in, run_name+'*'+hist_pref+'*.nc')

# List all files matching the pattern
files_sorted = sorted(glob.glob(files_unsorted))
print('-List of all h0* files')
print(files_sorted[0])
print(files_sorted[-1])

print()
print('-List of all files for requested years - '+str(years[0])+ ' to '+ str(years[1]))
files_in = [f for f in files_sorted if years[0] <= int(re.search(r'\d{4}', f).group()) <= years[1]]
print(files_in[0])
print(files_in[-1])

# File number check
print()
if len(files_in) != 12*(years[1]-years[0]+1): 
    print ('INCORRECT NUMBER OF FILES FOR YEARS REQUESTED -- ',years[0],' to ',years[1]) 
    sys.exit
else:
    print ('CORRECT NUMBER OF FILES FOR YEARS REQUESTED -- ',years[0],' to ',years[1]) 


-List of all h0* files
/glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/atm/hist/f.cam6_3_161.FLTHIST_ne30.ke.004.cam.h0a.1995-01.nc
/glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/atm/hist/f.cam6_3_161.FLTHIST_ne30.ke.004.cam.h0a.2012-12.nc

-List of all files for requested years - 1995 to 2004
/glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/atm/hist/f.cam6_3_161.FLTHIST_ne30.ke.004.cam.h0a.1995-01.nc
/glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/atm/hist/f.cam6_3_161.FLTHIST_ne30.ke.004.cam.h0a.2004-12.nc

CORRECT NUMBER OF FILES FOR YEARS REQUESTED --  1995  to  2004


In [6]:
'''
    Read in data (lazy)
'''

ds_hist = xr.open_mfdataset(files_in,parallel=True,chunks={"time": 12}) 
ds_hist = ds_hist.sel(time=slice(str(years[0]), str(years[1])))


In [7]:
'''
    Drop variables not needed and the string ones which cannot be grouped and averaged
'''

drop_vars_xtime = ['date_written','time_written','trop_cld_lev'] # Drop vars that are not time dimensioned, and add back in later if needed.

drop_vars_aer =['bc_c1','bc_c4','dst_c1','dst_c2','dst_c3','ncl_c1','ncl_c2','ncl_c3','num_c1','num_c2','num_c3','num_c4','pom_c1','pom_c4','so4_c1','so4_c2','so4_c3','soa_c1','soa_c2','bc_a1','bc_a4','dst_a1','dst_a2','dst_a3','ncl_a1','ncl_a2','ncl_a3','num_a1','num_a2','num_a3','so4_a1','so4_a2','so4_a3','soa_a1','soa_a2']

# Drop other unwanted 3D vars.
drop_vars_3d = ['ADRAIN','ADSNOW','ANSNOW','AWNI','AREI','AREL','AWNC','CCN3','CFC11','CFC12','CH4','CO2','DMS','GRAUQM','H2O2','H2SO4','N2O','NUMGRA','SNOWQM','SO2','SOAE','SOAG']
ds_vars = ds_hist.drop_vars(drop_vars_aer)
ds_vars = ds_vars.drop_vars(drop_vars_3d)
ds_vars = ds_vars.drop_vars(drop_vars_xtime)

ds_vars.attrs["History Directory"] = dir_in
ds_vars.attrs["Start Year"] = str(years[0])
ds_vars.attrs["End Year"] = str(years[1])
    


ds_vars

<xarray.Dataset> Size: 142GB
Dimensions:           (time: 120, lat: 192, lev: 58, ilev: 59, nbnd: 2, lon: 288)
Coordinates:
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 464B 3.018 5.445 9.087 ... 983.2 991.2 997.5
  * ilev              (ilev) float64 472B 2.055 3.98 6.909 ... 987.4 995.1 1e+03
  * time              (time) object 960B 1995-01-16 12:00:00 ... 2004-12-16 1...
Dimensions without coordinates: nbnd
Data variables: (12/324)
    w                 (time, lat) float64 184kB dask.array<chunksize=(1, 192), meta=np.ndarray>
    hyam              (time, lev) float64 56kB dask.array<chunksize=(1, 58), meta=np.ndarray>
    hybm              (time, lev) float64 56kB dask.array<chunksize=(1, 58), meta=np.ndarray>
    hyai              (time, ilev) float64 57kB dask.array<chunksize=(1, 59), meta=np.ndarray>
    hybi              (time, ilev) float64 57kB dask.array<chunksize=(1, 59), meta=np.ndarray>
    date              (time) int32 480B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                ...
    RTP2_CLUBB        (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    THLP2_CLUBB       (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    RTPTHLP_CLUBB     (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    WPRCP_CLUBB       (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    WPTHVP_CLUBB      (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    ZM_CLUBB          (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.cam6_3_161.FLTHIST_ne30.ke.004
    logname:            rneale
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FL...
    Start Year:         1995
    End Year:           2004

In [8]:
'''
 Calculate monthly climo - easy
'''

ds_cmonth =  ds_vars.groupby("time.month").mean()

# Month names for output file.
mname_file = ["%02d" % x for x in ds_cmonth.month]
#ds_cmonth.mean(dim='month')
ds_cmonth
ds_cmonth.attrs

{'interp_type': 'bilinear',
 'interp_outputgri': 'equally spaced with poles',
 'Conventions': 'CF-1.0',
 'source': 'CAM',
 'case': 'f.cam6_3_161.FLTHIST_ne30.ke.004',
 'logname': 'rneale',
 'host': 'dec0607',
 'initial_file': '/glade/campaign/cesm/cesmdata/inputdata/atm/cam/inic/se/FLT_L58_ne30pg3_IC_c220623.nc',
 'topography_file': '/glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/se/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0100_noleak_20240117.nc',
 'model_doi_url': 'not_set',
 'time_period_freq': 'month_1',
 'History Directory': '/glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/atm/hist/',
 'Start Year': '1995',
 'End Year': '2004'}

In [9]:
print('-Calculate annual climatology...')

ds_cyear =  ds_vars.mean(dim='time')

# Month names for output file.
#mname_file = ["%02d" % x for x in ds_cmonth.month]
ds_cyear

-Calculate annual climatology...


<xarray.Dataset> Size: 1GB
Dimensions:           (lat: 192, lev: 58, ilev: 59, lon: 288)
Coordinates:
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 464B 3.018 5.445 9.087 ... 983.2 991.2 997.5
  * ilev              (ilev) float64 472B 2.055 3.98 6.909 ... 987.4 995.1 1e+03
Data variables: (12/323)
    w                 (lat) float64 2kB dask.array<chunksize=(192,), meta=np.ndarray>
    hyam              (lev) float64 464B dask.array<chunksize=(58,), meta=np.ndarray>
    hybm              (lev) float64 464B dask.array<chunksize=(58,), meta=np.ndarray>
    hyai              (ilev) float64 472B dask.array<chunksize=(59,), meta=np.ndarray>
    hybi              (ilev) float64 472B dask.array<chunksize=(59,), meta=np.ndarray>
    date              float64 8B dask.array<chunksize=(), meta=np.ndarray>
    ...                ...
    RTP2_CLUBB        (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    THLP2_CLUBB       (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    RTPTHLP_CLUBB     (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    WPRCP_CLUBB       (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    WPTHVP_CLUBB      (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    ZM_CLUBB          (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>

In [10]:
print('-Calculate weighted seasonal climatology...')

days_in_month = ds_vars.time.dt.days_in_month

mweights = (days_in_month.groupby('time.season') / days_in_month.groupby('time.season').sum())

-Calculate weighted seasonal climatology...


In [11]:
# Calculate weighted seasonal averages

ds_wcseas = ((ds_vars*mweights).groupby("time.season").sum(dim='time'))
ds_wcseas = ds_wcseas.astype(np.float32)

In [12]:
# Need to copy the attributes due to the array multiplcation wiping them out.

# Global 

ds_wcseas.attrs = ds_vars.attrs 

# Variable

for var in ds_wcseas.data_vars:
    ds_wcseas[var].attrs = ds_vars[var].attrs



In [13]:
'''
    Writing out climo. files
'''

# Check out dir exists
if not os.path.exists(dir_out): os.makedirs(dir_out)

print('-Writing out to directory ...')

# Writing out monthly climatologies

for imm,mname in enumerate(ds_cmonth.month.values):
    
    fout_mon = dir_out+run_name+'_'+mname_file[imm]+'_climo.nc'
    print(mname_file[imm],' -- Writing - ',fout_mon)

    ds_cmonth.sel(month=mname).to_netcdf(fout_mon)
    print('-Done...')

# Writing out seasonal climatologies

for sname in ds_wcseas.season.values:
    fout_seas = dir_out+run_name+'_'+sname+'_climo.nc'
    print(sname)
    print(sname,'-- Writing - ',fout_seas)
    ds_wcseas.sel(season=sname).drop_vars('season').to_netcdf(fout_seas)
    print('-Done...')

# Wrting out annual climatology
fout_ann = dir_out+run_name+'_ANN_climo.nc'

print('-Writing - ',fout_ann)
ds_cyear.to_netcdf(fout_ann)
print('-Done...')

print('---- COMPLETE ----')



-Writing out to directory ...
01  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/climo/f.cam6_3_161.FLTHIST_ne30.ke.004_01_climo.nc
-Done...
02  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/climo/f.cam6_3_161.FLTHIST_ne30.ke.004_02_climo.nc
-Done...
03  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/climo/f.cam6_3_161.FLTHIST_ne30.ke.004_03_climo.nc
-Done...
04  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/climo/f.cam6_3_161.FLTHIST_ne30.ke.004_04_climo.nc
-Done...
05  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/climo/f.cam6_3_161.FLTHIST_ne30.ke.004_05_climo.nc
-Done...
06  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST_ne30.ke.004/climo/f.cam6_3_161.FLTHIST_ne30.ke.004_06_climo.nc
-Done...
07  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_3_161.FLTHIST

RuntimeError: NetCDF: Not a valid ID